In [ ]:
REPO_URL = 'https://github.com/11pingupingu99-max/UMUD_Kaggle_Bruh_submission'
# CPU wystarczy. Nie wykonuje treningu, inferencji ani publikacji.
DRIVE_ROOT = '/content/drive/MyDrive'
OUTPUT_DIR = None  # domyślnie MyDrive/UMUD_C1_WEIGHTS
MAKE_ZIP = True

**Repository:** [https://github.com/11pingupingu99-max/UMUD_Kaggle_Bruh_submission](https://github.com/11pingupingu99-max/UMUD_Kaggle_Bruh_submission)

# UMUD — zebranie finalnych 30 checkpointów

Otwórz w Colabie na **CPU** i uruchom wszystkie komórki. Narzędzie zbiera dokładnie wagi z pełnego biegu C1,
sprawdza SHA-256 każdego pliku i układa je do użycia przez finalny notebook Kaggle. Zajmuje kilka GB miejsca.
Kopiowanie można wznowić; istniejący plik z inną sumą przerywa pracę. Nie zastępuje go po cichu.

Po zakończeniu użyj **rozpakowanej zawartości `UMUD_C1_WEIGHTS`** do utworzenia Kaggle Dataset,
a następnie dodaj go jako Input notebooka `UMUD_FINAL_KAGGLE.ipynb`. Do czasu publikacji Dataset może być prywatny.
Publikacja repo i wag jest oddzielnym krokiem; ten notebook niczego nie publikuje i nie wymaga klucza Kaggle.

In [ ]:
# Jawny pakiet źródeł; brak CSV predykcji, etykiet hosta i binarnych wag.
import base64, hashlib, io, json, stat, sys, zipfile
from pathlib import Path, PurePosixPath
WORK = Path('/kaggle/working') if Path('/kaggle').is_dir() else Path('/content')
RUNTIME = WORK / 'umud_final_runtime_20261114'
RUNTIME.mkdir(parents=True, exist_ok=True)
PAYLOAD_SHA256 = '48924ca0f6924af2757aaa815c21580ae534ee7425d30926965f45f81ca5b9ce'
PAYLOAD_B64 = '...'
raw = base64.b64decode(PAYLOAD_B64, validate=True)
if hashlib.sha256(raw).hexdigest() != PAYLOAD_SHA256:
    raise ValueError('Niezgodny pakiet źródeł')
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for item in archive.infolist():
        rel = PurePosixPath(item.filename)
        if rel.is_absolute() or '..' in rel.parts or stat.S_ISLNK(item.external_attr >> 16):
            raise ValueError('Niepoprawna ścieżka w archiwum źródeł')
        target = RUNTIME / str(rel)
        data = archive.read(item)
        if target.exists() and target.read_bytes() != data:
            raise ValueError(f'Inna wersja źródeł w {target}; użyj nowego runtime')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
for rel, expected in json.loads((RUNTIME / 'RUNTIME_FILES_SHA256.json').read_text()).items():
    if hashlib.sha256((RUNTIME / rel).read_bytes()).hexdigest() != expected:
        raise ValueError(f'Niezgodne źródło: {rel}')
sys.path.insert(0, str(RUNTIME / 'tools'))
import final_notebook_support as support
if support.frozen_source_hash(RUNTIME) != support.SOURCE_SHA256:
    raise ValueError('Kod matematyczny lub sieciowy różni się od pełnego biegu C1')
print('Źródła C1 zweryfikowane:', support.SOURCE_SHA256)

In [ ]:
from google.colab import drive
if not Path(DRIVE_ROOT).is_dir():
    drive.mount('/content/drive')
import subprocess, shutil
OUTPUT_DIR = Path(OUTPUT_DIR) if OUTPUT_DIR else Path(DRIVE_ROOT) / 'UMUD_C1_WEIGHTS'
PROOF = WORK / 'umud_model_proof_20261114'
PROOF.mkdir(parents=True, exist_ok=True)
registry = json.loads((RUNTIME / 'models/DRIVE_ORIGINAL_REGISTRY.json').read_text())
for family, entries in registry.items():
    for d in entries:
        rel = Path(d['path']).relative_to('/content/drive/MyDrive')
        d['path'] = str(Path(DRIVE_ROOT) / rel)
for filename in ('MODELS_ORIGINAL.json', 'MODELS_VERIFIED.json'):
    (PROOF / filename).write_text(json.dumps(registry, indent=2))
shutil.copyfile(RUNTIME / 'models/AUDITED_RUN_MODEL_SIGNATURE.json', PROOF / 'DONE.json')
subprocess.run([sys.executable, '-u', str(RUNTIME / 'tools/export_weights.py'),
                '--run-dir', str(PROOF), '--output-dir', str(OUTPUT_DIR)], check=True)
complete = json.loads((OUTPUT_DIR / 'WEIGHTS_COMPLETE.json').read_text())
if complete['model_count'] != 30:
    raise ValueError('Niepełna paczka wag')
print('Sprawdzono 30/30 wag. Łącznie GB:', round(complete['total_bytes'] / 1e9, 3))
print('Katalog gotowy do utworzenia Datasetu:', OUTPUT_DIR)

In [ ]:
# ZIP zawiera wyłącznie wymienione checkpointy i metadane tego pakietu.
if MAKE_ZIP:
    import os
    archive_path = OUTPUT_DIR.parent / (OUTPUT_DIR.name + '.zip')
    temp = archive_path.with_suffix('.zip.partial')
    items = [d['file'] for d in complete['files']] + [
        'WEIGHTS_MANIFEST.json', 'model_registry.json', 'WEIGHTS_COMPLETE.json', 'README.md']
    with zipfile.ZipFile(temp, 'w', zipfile.ZIP_STORED, allowZip64=True) as z:
        for name in items:
            z.write(OUTPUT_DIR / name, OUTPUT_DIR.name + '/' + name)
    os.replace(temp, archive_path)
    receipt = {'archive': archive_path.name, 'bytes': archive_path.stat().st_size,
               'sha256': support.sha256(archive_path), 'checkpoint_count': 30,
               'publication_status': 'LOCAL_PACKAGE_NOT_UPLOADED'}
    archive_path.with_suffix('.zip.json').write_text(json.dumps(receipt, indent=2))
    print('Paczka ZIP:', archive_path)
    print(json.dumps(receipt, indent=2))
print('Dodaj do Kaggle rozpakowane pliki z zachowaniem podkatalogów r12b, r22, v2, a76, f71, f86, r17.')
print('Następnie w finalnym notebooku wybierz GPU i Add Input → ten Dataset oraz dane konkursu.')